In [6]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [1]:
from langchain_core.documents import Document

In [2]:
sample_doc = Document(
    page_content='RAG project',
    metadata ={'source':"https://www.google.com"}
)

In [3]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='RAG project')

In [4]:
#PDF data
from langchain_community.document_loaders.pdf import PyPDFLoader  #also use PyMuPDFLoader

pdf_loader = PyPDFLoader('pdf_files/KNN - K Nearest Neighbors.pdf')

In [5]:
document = pdf_loader.load()

document

[Document(metadata={'producer': 'pdfcpu v0.9.1 dev', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-04-16T11:11:14+00:00', 'author': 'Studio', 'moddate': '2026-04-16T11:11:14+00:00', 'title': 'Linear Regression (Notes)', 'source': 'pdf_files/KNN - K Nearest Neighbors.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content="1  \nKNN - K Nearest Neighbors APNA COLLEGE \nKNN - K Nearest Neighbors  \n \n \nIntuition & Logic \n \nKNN (K-Nearest Neighbors) is a supervised ML algorithm used for classification and \nregression problems. \n \nKNN makes predictions by looking at the K closest data points to a new data point \nand using their information. Basically on the logic - “Tell me who your neighbors are, \nand I’ll tell you who you are.” \n \n \nLet’s suppose we create a scatter plot of data split into 2 categories: \n \n \n \nHow do we predict which class the new data point belongs to? We use KNN. \n \nHow does KNN work? \n \n1. We choose an odd number as K (like 3,

## Ingestion pipeline

In [6]:
#Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [7]:
def load_all_pdfs():
  folder_path = 'pdf_files'
  num_docs=0
  all_docs =[]

  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      #complete file path
      pdf_path = os.path.join(folder_path,filename)

      loader = PyPDFLoader(pdf_path)
      doc = loader.load()

      all_docs.extend(doc)
      num_docs +=1

  print("total pdfs:",num_docs)
  print("total pages:",len(all_docs))
  return all_docs

In [8]:
all_pdf_documents = load_all_pdfs()

total pdfs: 3
total pages: 24


In [9]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

In [28]:
#chunks
# !pip install langchain_text_splitters

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents,chunk_size=500,chunk_overlap=50):

  text_splitter =RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunked_docs = text_splitter.split_documents(documents)
  return chunked_docs

In [11]:
chunks = split_doc(all_pdf_documents)

In [12]:
len(chunks)

75

## Chunks

In [13]:
from sentence_transformers import SentenceTransformer

In [14]:
class EmeddingManager:
  def __init__(self,model_name="all-MiniLM-L6-v2"):
    self.model_name = model_name
    print("Loading...",self.model_name)
    self.model= SentenceTransformer(model_name)
    print("Embedding dimensions=",self.model.get_sentence_embedding_dimension())

  def generate_embeddings(self,text):
    embeddings = self.model.encode(text,show_progress_bar=True)
    print("embedding shape",embeddings.shape)
    return embeddings

In [15]:
embedding_manager = EmeddingManager()

Loading... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimensions= 384


C:\Users\sonu2\AppData\Local\Temp\ipykernel_7808\3768426719.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimensions=",self.model.get_sentence_embedding_dimension())


In [35]:
# !pip install chromadb uuid

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for uuid: filename=uuid-1.30-py3-none-any.whl size=6485 sha256=a90fda71c8e7c7faf4fdb918f31b5e8e4b17338681d3d0e2fc88f2c9d2d24838
  Stored in directory: c:\users\sonu2\appdata\local\pip\cache\wheels\cc\9d\72\13ff6a181eacfdbd6651ed761a4ee7c5c9f92034a9dc8a1b3c
Successfully built uuid


  DEPRECATION: Building 'uuid' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'uuid'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [16]:
import uuid
import chromadb

In [17]:
class VectorStoreManager:
  def __init__(self,persist_directory="RAG-project/Vector_store", collection_name="pdf_documents"):
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.collection = None
    self.client = None

  def _initialize_store(self):
    os.makedirs(self.persist_directory,exist_ok=True)

    #Create a client
    self.client = chromadb.PersistentClient(path=self.persist_directory)

    #Create the collection
    self.collection = self.client.get_or_create_collection(
        name=self.collection_name,
        metadata={"description":"vector store collection for pdf embeddings in RAG"}
    )

    print("Initialize the vectore store with collection:",self.collection_name)
    print("docs in collection:",self.collection.count())

  def add_documents(self,documents,embeddings):
    if len(documents) != len(embeddings):
      raise ValueError("num of documents does not match num of embeddings")

    #store => ids,embeddings,document,metadata
    ids = []
    all_metadata =[]
    documents_content = []
    embeddings_list = []

    for i ,(doc,embedding) in enumerate(zip(documents,embeddings)):
      doc_id = f"doc_{uuid.uuid4()}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_length'] = len(doc.page_content)
      all_metadata.append(metadata)

      documents_content.append(doc.page_content)

      embeddings_list.append(embedding.tolist())

      self.collection.add(
          ids=ids,
          metadatas=all_metadata,
          documents=documents_content,
          embeddings=embeddings_list
      )

    print("Total documents added in vector store:",len(documents_content))
    print("docs in collection:",self.collection.count())


In [26]:
vector_store = VectorStoreManager()
vector_store._initialize_store()

vector_store.add_documents(chunks, embeddings) 

Initialize the vectore store with collection: pdf_documents
docs in collection: 145
Total documents added in vector store: 75
docs in collection: 220


In [19]:
uuid.uuid4()

UUID('e8a3254d-b93f-4ad4-8eca-f08e7cb0a39b')

## Retrieval pipeline

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

In [21]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self,query,top_k=5,score_threshold=0.0):
        #uery => embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        
        results = self.vector_store.collection.query(
            query_embeddings = [query_embedding.tolist()],
            n_results = top_k
        )

        #cosine similarity
        retrieved_docs = []
        if results['documents'] and results['documents'][0]:
            ids = results['ids'][0]
            metadatas = results['metadatas'][0]
            documents = results['documents'][0]
            distances = results['distances'][0]

            for i,(doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score = 1-distance
                
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity": similarity_score,
                        "rank": i+1
                    })

            print(f"retrived {len(retrieved_docs)} documents")

        else:
            print("No documents Found")

        return retrieved_docs

In [22]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [23]:
rag_retriever.retrieve("KNN")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrived 5 documents


[{'id': 'doc_ef836c12-9882-4399-9e2c-f3208ed091e5',
  'document': '1  \nKNN - K Nearest Neighbors APNA COLLEGE \nKNN - K Nearest Neighbors  \n \n \nIntuition & Logic \n \nKNN (K-Nearest Neighbors) is a supervised ML algorithm used for classification and \nregression problems. \n \nKNN makes predictions by looking at the K closest data points to a new data point \nand using their information. Basically on the logic - “Tell me who your neighbors are, \nand I’ll tell you who you are.” \n \n \nLet’s suppose we create a scatter plot of data split into 2 categories:',
  'metadata': {'title': 'Linear Regression (Notes)',
   'moddate': '2026-04-16T11:11:14+00:00',
   'page_label': '1',
   'doc_index': 16,
   'producer': 'pdfcpu v0.9.1 dev',
   'creator': 'Microsoft® Word LTSC',
   'content_length': 484,
   'page': 0,
   'author': 'Studio',
   'source': 'pdf_files\\KNN - K Nearest Neighbors.pdf',
   'total_pages': 2,
   'creationdate': '2026-04-16T11:11:14+00:00'},
  'distance': 0.8012627363204

In [24]:
print("Docs in collection:", vector_store.collection.count())

Docs in collection: 70


In [25]:
# Step 1 — extract text from chunks
all_texts = [chunk.page_content for chunk in chunks]

# Step 2 — generate embeddings
embeddings = embedding_manager.generate_embeddings(all_texts)

# Step 3 — verify
print("Number of chunks:", len(chunks))
print("Embeddings shape:", embeddings.shape)

# Step 4 — now add to vector store
vector_store.add_documents(chunks, embeddings)

# Step 5 — confirm
print("Docs in collection:", vector_store.collection.count())

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

embedding shape (75, 384)
Number of chunks: 75
Embeddings shape: (75, 384)
Total documents added in vector store: 75
docs in collection: 145
Docs in collection: 145


## Integrate RAG with LLMs

# 1.OpenAI - GPT

In [ ]:
API_KEY_OPENAI = "YOUR_OPENAI_API_KEY"

In [27]:
# !pip install langchain openai

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.2 MB ? eta -:--:--
   ------------------------------------ --- 1.0/1.2 MB 3.5 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 MB 3.2 MB/s eta 0:00:00
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)

  Attempting uninstall: typing-extensions

    Found existing installation: typing_extensions 4.12.2

    Uninstalling typing_extensions-4.12.2:

   ---------------------------------------- 0/3 [typing-extensions]
   ---------------------------------------- 0/3 [typing-extensions]
   ---------------------------------------- 0/3 [typing-extensions]
   ---------------------------------------- 0/3 [typing-extensions]
   ---------------------------------------- 0/3 [typing-extensions]
   ---------------------------------------- 0/3 [typing-extensions]
   --------------

In [53]:
#pip install -U langchain langchain-google-genai chromadb

In [30]:
# !pip install -U langchain-openai

   ---------------------------------------- 0.0/879.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/879.1 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/879.1 kB ? eta -:--:--
   ----------------------- ---------------- 524.3/879.1 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 879.1/879.1 kB 1.3 MB/s eta 0:00:00

   -------------------- ------------------- 1/2 [langchain-openai]
   ---------------------------------------- 2/2 [langchain-openai]



In [55]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [56]:
# generate our retrieval-agumented output
def generate_output(query,rag_retriever,llm,top_k=3):
    results = rag_retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("WE found no relevent context for the given query...")

    #Content + query
    prompt = f""" Use given context to generate the answer for the query
                  Context:{context}
                  Query:{query} """

    response = llm.invoke(prompt) #expecting a string as prompt
    return response.content

In [69]:
# this platform is paid so the output was not generated
answer = generate_output("what are Reinforcement components?", rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrived 3 documents


## 2.GROQ

In [ ]:
API_KEY_GROQ = "your_groq_api_key"

In [60]:
# !pip install langchain-groq


   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   ---------------------------------------- 0/2 [groq]
   -------------------- ------------------- 1/2 [langchain-groq]
   ---------------------------------------- 2/2 [langchain-groq]



In [63]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [66]:
#generate our retrieval-augmented output
def generate_output(query,rag_retiever,llm,top_k=3):
    results= rag_retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"]for doc in results]) if results else ""

    if not context:
        print("We found no relevent context for the given query")

    #context + query
    prompt = f""" Use given context to generate the answer for the query
                  Context: {context}
                  Query: {query} """
    response = llm.invoke([prompt.format(context=context,query=query)]) # expecting a list as prompt
    return response.content

In [67]:
answer = generate_output("What is reinforcement learning",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrived 3 documents


In [68]:
print(answer)

<think>
Okay, the user is asking, "What is reinforcement learning?" Let me look at the context provided.

The context starts by mentioning Reinforcement Learning (RL) as a type of Machine Learning where an agent learns to make decisions by interacting with an environment. It contrasts RL with supervised learning by noting there are no labeled examples. The agent learns through trial and error, getting rewards for good actions and penalties for bad ones. The core idea is to maximize cumulative reward over time.

I need to make sure I capture all these points. The user probably wants a concise definition that includes the main components: agent, environment, learning through interaction, rewards/penalties, and the goal of maximizing cumulative reward. Also, the context is repeated a few times, so I should avoid redundancy. Maybe mention that it's different from supervised learning because there are no labeled examples. Let me structure the answer clearly, starting with the definition, th

# 3. Gemini

In [85]:
# !pip install --upgrade langchain-google-genai google-generativeai
# !pip install "google-genai>=0.8.0,<1.0.0"  # pin to stable version

INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.3 MB ? eta -:--:--
   --------------- ------------------------ 0.5/1.3 MB 990.6 kB/s eta 0:00:01
   --------------- ------------------------ 0.5/1.3 MB 990.6 kB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.3 MB 608.1 kB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.3 MB 608.1 kB/s eta 0:00:01
   ----------------------- ---------------- 0.8/1.3 MB 608.1 kB/s eta 0:00:01
   -------------------------

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


In [90]:
import subprocess
subprocess.run(["pip", "uninstall", "google-genai", "langchain-google-genai", "-y"])
subprocess.run(["pip", "install", "google-generativeai==0.8.3", "langchain-google-genai==2.1.4", "--quiet"])

CompletedProcess(args=['pip', 'install', 'google-generativeai==0.8.3', 'langchain-google-genai==2.1.4', '--quiet'], returncode=1)

In [ ]:
API_KEY_GEMINI = "YOUR_GEMINI_API_KEY"

In [91]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    google_api_key=API_KEY_GEMINI,
    model="gemini-1.5-flash",
    temperature=1.0,
    max_tokens=1024
)

ModuleNotFoundError: No module named 'langchain_google_genai'

In [78]:
#generate our retrieval-augmented output
def generate_output(query,rag_retiever,llm,top_k=3):
    results= rag_retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"]for doc in results]) if results else ""

    if not context:
        print("We found no relevent context for the given query")

    #context + query
    prompt = f""" Use given context to generate the answer for the query
                  Context: {context}
                  Query: {query} """
    response = llm.invoke([prompt.format(context=context,query=query)]) # expecting a list as prompt
    return response.content